## Step 1: Load Splits and Define Available Features
Load train, validation, and test splits. Identify which columns are actually available at prediction time (order purchase moment) — excluding any post-delivery information like actual delivery dates or review scores, which would cause data leakage.

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("artifacts/train.csv", parse_dates=["order_purchase_timestamp"])
val = pd.read_csv("artifacts/val.csv", parse_dates=["order_purchase_timestamp"])
test = pd.read_csv("artifacts/test.csv", parse_dates=["order_purchase_timestamp"])

print(f"Train: {train.shape} | Val: {val.shape} | Test: {test.shape}")

# Columns NOT available at prediction time (leakage risk)
leakage_cols = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "review_score",
    "n_reviews",
]
print(f"\nExcluded (leakage) columns: {leakage_cols}")

Train: (69608, 21) | Val: (14916, 21) | Test: (14917, 21)

Excluded (leakage) columns: ['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'review_score', 'n_reviews']


## Step 2: Keep Only Labeled Rows
Drop rows where `is_late` is missing (orders that were never delivered) — these can't be used for training or evaluation.

In [2]:
train = train[train["is_late"].notna()].copy()
val = val[val["is_late"].notna()].copy()
test = test[test["is_late"].notna()].copy()

print(f"Train: {train.shape} | Val: {val.shape} | Test: {test.shape}")

Train: (67222, 21) | Val: (14674, 21) | Test: (14574, 21)


## Step 3: Build Base Features
Select the features to use, based on EDA findings: customer_state (strong geographic effect), numerical order features, and a seasonal feature (order month) since we found strong seasonality in the late-delivery rate.

In [3]:
def build_features(df):
    df = df.copy()
    df["order_month"] = df["order_purchase_timestamp"].dt.month
    df["order_weekday"] = df["order_purchase_timestamp"].dt.weekday
    
    feature_cols = [
        "customer_state", "customer_zip_code_prefix",
        "n_items", "total_price", "total_freight",
        "total_payment_value", "n_payments", "max_installments",
        "order_month", "order_weekday"
    ]
    return df[feature_cols + ["is_late"]]

train_feat = build_features(train)
val_feat = build_features(val)
test_feat = build_features(test)

print(train_feat.shape)
train_feat.head()

(67222, 11)


,customer_state,customer_zip_code_prefix,n_items,total_price,total_freight,total_payment_value,n_payments,max_installments,order_month,order_weekday,is_late
3,SP,14600,3.0,134.97,8.49,NaN,NaN,NaN,9,3,1.0
5,SP,4106,1.0,29.90,15.56,45.46,1.0,1.0,10,0,0.0
6,RS,98280,1.0,21.90,17.19,39.09,1.0,1.0,10,0,0.0
8,RS,90040,1.0,36.49,17.24,53.73,1.0,1.0,10,0,0.0
9,SP,13185,1.0,119.90,13.56,133.46,1.0,6.0,10,0,0.0


## Step 4: Handle Missing Values (Imputation)
Fit imputers on the training data only, then apply the same fitted imputers to validation and test — never re-fit on new data.

In [4]:
from sklearn.impute import SimpleImputer

numerical_cols = ["n_items", "total_price", "total_freight", 
                   "total_payment_value", "n_payments", "max_installments"]

# Fit imputer on TRAIN only
num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(train_feat[numerical_cols])

# Apply to train, val, test
for df in [train_feat, val_feat, test_feat]:
    df[numerical_cols] = num_imputer.transform(df[numerical_cols])

print("Missing values after imputation:")
print(train_feat[numerical_cols].isna().sum())

Missing values after imputation:
n_items                0
total_price            0
total_freight          0
total_payment_value    0
n_payments             0
max_installments       0
dtype: int64


## Step 5: Encode Categorical Features
Encode `customer_state` using one-hot encoding. Fit the encoder on training data only, then apply to validation and test.

In [5]:
from sklearn.preprocessing import OneHotEncoder

state_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
state_encoder.fit(train_feat[["customer_state"]])

def apply_encoding(df):
    encoded = state_encoder.transform(df[["customer_state"]])
    encoded_cols = state_encoder.get_feature_names_out(["customer_state"])
    encoded_df = pd.DataFrame(encoded, columns=encoded_cols, index=df.index)
    df = df.drop(columns=["customer_state"]).reset_index(drop=True)
    encoded_df = encoded_df.reset_index(drop=True)
    return pd.concat([df, encoded_df], axis=1)

train_encoded = apply_encoding(train_feat)
val_encoded = apply_encoding(val_feat)
test_encoded = apply_encoding(test_feat)

print(f"Train encoded shape: {train_encoded.shape}")
train_encoded.head()

Train encoded shape: (67222, 37)


,customer_zip_code_prefix,n_items,total_price,total_freight,total_payment_value,n_payments,max_installments,order_month,order_weekday,is_late,...,customer_state_PR,customer_state_RJ,customer_state_RN,customer_state_RO,customer_state_RR,customer_state_RS,customer_state_SC,customer_state_SE,customer_state_SP,customer_state_TO
0,14600,3.0,134.97,8.49,104.12,1.0,2.0,9,3,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,4106,1.0,29.90,15.56,45.46,1.0,1.0,10,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,98280,1.0,21.90,17.19,39.09,1.0,1.0,10,0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,90040,1.0,36.49,17.24,53.73,1.0,1.0,10,0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,13185,1.0,119.90,13.56,133.46,1.0,6.0,10,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## Step 6: Scale Numerical Features
Fit a scaler on training data only, then apply to validation and test.

In [6]:
from sklearn.preprocessing import StandardScaler

scale_cols = ["customer_zip_code_prefix", "n_items", "total_price", "total_freight",
              "total_payment_value", "n_payments", "max_installments",
              "order_month", "order_weekday"]

scaler = StandardScaler()
scaler.fit(train_encoded[scale_cols])

for df in [train_encoded, val_encoded, test_encoded]:
    df[scale_cols] = scaler.transform(df[scale_cols])

print("✅ Scaling applied")
train_encoded[scale_cols].describe().round(2)

✅ Scaling applied


,customer_zip_code_prefix,n_items,total_price,total_freight,total_payment_value,n_payments,max_installments,order_month,order_weekday
count,67222.00,67222.00,67222.00,67222.00,67222.00,67222.00,67222.00,67222.00,67222.00
mean,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.17,-0.26,-0.65,-1.12,-0.69,-0.12,-0.72,-1.32,-1.41
25%,-0.80,-0.26,-0.44,-0.41,-0.45,-0.12,-0.72,-0.79,-0.91
50%,-0.34,-0.26,-0.25,-0.27,-0.25,-0.12,-0.35,-0.26,0.11
75%,0.81,-0.26,0.07,0.07,0.08,-0.12,0.37,1.07,0.62
max,2.14,36.70,64.79,49.18,62.99,62.70,7.63,1.60,1.64


## Step 7: Save Artifacts
Save the final feature tables (train/val/test) and all fitted transformers (imputer, encoder, scaler) so the production pipeline can reuse them without re-fitting on new data.

In [7]:
import joblib

# Save feature tables
train_encoded.to_csv("artifacts/train_features.csv", index=False)
val_encoded.to_csv("artifacts/val_features.csv", index=False)
test_encoded.to_csv("artifacts/test_features.csv", index=False)

# Save fitted transformers
joblib.dump(num_imputer, "artifacts/num_imputer.joblib")
joblib.dump(state_encoder, "artifacts/state_encoder.joblib")
joblib.dump(scaler, "artifacts/scaler.joblib")

# Save feature list
feature_list = [c for c in train_encoded.columns if c != "is_late"]
with open("artifacts/feature_list.txt", "w") as f:
    f.write("\n".join(feature_list))

print(f"✅ Saved feature tables, transformers, and feature list ({len(feature_list)} features)")

✅ Saved feature tables, transformers, and feature list (36 features)
